# CS781 Phase 4 — Court Judgment Prediction (CJPE): fine-tune the Phase 1 models and generate the six submission files

**Task:** given the text of an Indian Supreme Court case, predict the outcome as the integer 0 or 1. **Metric:** macro-F1 (primary), precision and recall. **Deadline:** 21 Oct, 11:59 PM.
**Data** (gated, Hugging Face `Exploration-Lab/CS-781-Capstone`): `Phase_4_CJPE/Train/train.json` (737 MB), `Train/dev.json` (80 MB), `Test/test.json` (35 MB). The sizes suggest roughly 32k / 3.5k / 1.5k cases of about 23 KB each (an inference from the file sizes; cell 2 prints the real counts).

## How this notebook follows the rules
| Rule | What the notebook does |
|---|---|
| start from your own Phase 1 model; use **all four** architectures | cell 3 loads the decoder (GPT-2), encoder (ModernBERT-style), encoder-decoder (T5 encoder) and recurrent (BiLSTM) models, plus the old 3,000-step GPT-2, through `arch_loaders.py` |
| six best experiments involving **different** Phase 1 models | cell 6 defines six experiments across five distinct Phase 1 checkpoints; cell 7 picks the best experiment of every distinct model first, warns if an architecture is missing |
| classification head | `DocClassifier`: pooled chunk vectors -> attention pooling + final chunk -> small MLP -> 2 classes (class-weighted cross-entropy for macro-F1) |
| long documents | the **end** of the case is used (published CJPE work finds the end most informative): `K` chunks of `L` tokens, right-aligned; `K=1` is "last L tokens", `K>1` is hierarchical |
| train only on the provided data | yes: only `train.json` for training; `dev.json` only to choose the snapshot and the decision threshold |
| submission format | `[{"id": ..., "pred": 0|1}, ...]`, ids exactly as in `test.json`, validated before saving |

## Inputs to attach (Add Input)
1. Your Phase 1 dataset with `arch_loaders.py`, `legal_bpe_32k_mask.json` and the GPT-2 model (`cs781-phase1-v3-gpt2-decoder`).
2. The pre-trained encoder, T5 and BiLSTM models (the outputs or datasets of the pre-training notebooks; any folder containing a `final_model.pt`). They are found automatically.
3. Your old Phase 1 checkpoint dataset (`cs781-phase-1-checkpoints`) for the old GPT-2 experiment.
4. The CJPE data: attach it as a dataset, or let cell 2 download it (needs a Hugging Face token: a Kaggle secret named `HF_TOKEN`, or a hidden prompt in an interactive session).

## Run order
1 setup -> 2 data (first run tokenises about 850 MB of text on the CPU, then caches) -> 3 models -> 4 trainer -> **5 smoke test** -> 6 experiments (each in its own cell; run what is available) -> 7 pick the six, validate and package.
Experiments are budgeted in **minutes** (cell 6): the schedule is sized to your real GPU speed. Default total is about 3.4 GPU-hours for the six.

## What was tested
On CPU with tiny models and synthetic data in the real file layout: schema detection, caching, integer and string ids, right-aligned chunking (chunks concatenate back to the last K*L tokens), all four architectures plus an old-format checkpoint, threshold search, the submission validator (rejects bool/float/2, wrong count, wrong id), and the final packaging. A planted signal in the last tokens was learned perfectly by all six experiments; a planted signal 80-110 tokens before the end was missed by the last-chunk model (F1 0.47) and found by the hierarchical model (F1 1.00).
**Not tested:** the real CJPE files, GPU speed and memory, and real-size model quality. Expect much lower scores than the synthetic ones; published models with strong pre-training reach about 76-78% macro-F1 on this kind of data, and your models are small and lightly pre-trained.


In [ ]:
# ================= CELL 1 — setup =================
import os, sys, re, json, math, time, random, glob, gc, copy, hashlib, pickle, getpass, zipfile
from pathlib import Path
from collections import Counter
import numpy as np, pandas as pd, torch, torch.nn as nn, torch.nn.functional as F

SEED = 42
def seed_all(s):
    random.seed(s); np.random.seed(s); torch.manual_seed(s)
    if torch.cuda.is_available(): torch.cuda.manual_seed_all(s)
seed_all(SEED)
device = torch.device("cuda" if torch.cuda.is_available() else "cpu"); AMP = device.type == "cuda"
WORK = Path("/kaggle/working/cjpe"); (WORK / "runs").mkdir(parents=True, exist_ok=True); (WORK / "cache").mkdir(exist_ok=True); SUBDIR = WORK / "submissions"; SUBDIR.mkdir(exist_ok=True)
LEDGER = WORK / "ledger.csv"
SEARCH_ROOTS = ["/kaggle/input", "/kaggle/working"]
print("device:", torch.cuda.get_device_name(0) if device.type == "cuda" else "cpu (training cells need a GPU)", "| work dir:", WORK)


In [ ]:
# ================= CELL 2 — CJPE data: locate (or download), detect the schema, tokenise ONLY what is needed (the end of each case), cache =================
HF_REPO = "Exploration-Lab/CS-781-Capstone"          # gated: Phase_4_CJPE/Train/train.json (737 MB), Train/dev.json (80 MB), Test/test.json (35 MB)
TAIL_MAX = 4096                                      # tokens kept from the END of every case (the end of a judgment is the most informative part)
TEXT_KEYS = ("text", "document", "case_text", "content", "judgment", "doc", "full_text", "case"); LABEL_KEYS = ("label", "labels", "decision", "outcome", "y", "gold", "judgement", "verdict")
ID_KEYS = ("id", "case_id", "doc_id", "document_id", "name", "file", "filename", "index")

def locate():
    got = {}
    for sp in ("train", "dev", "test"):
        c = [p for root in SEARCH_ROOTS for p in glob.glob(f"{root}/**/{sp}.json", recursive=True) if re.search(r"cjpe|phase[_ -]?4", p, re.I)]
        if c: got[sp] = max(c, key=os.path.getsize)
    return got
paths = locate()
if len(paths) < 3:
    print("CJPE files not found locally -> downloading from Hugging Face (accept the dataset conditions on its page first)")
    from huggingface_hub import snapshot_download
    token = os.environ.get("HF_TOKEN")
    if not token:
        try:
            from kaggle_secrets import UserSecretsClient; token = UserSecretsClient().get_secret("HF_TOKEN")
        except Exception: token = getpass.getpass("Hugging Face token (hidden; not stored): ").strip()
    local = snapshot_download(HF_REPO, repo_type="dataset", allow_patterns=["Phase_4_CJPE/*"], token=token, local_dir=str(WORK / "hf")); SEARCH_ROOTS.append(local); paths = locate()
assert len(paths) == 3, f"need train/dev/test json, found {paths}"
print({k: f"{os.path.getsize(v)/2**20:.0f} MiB  {v}" for k, v in paths.items()})

def normalise(obj):
    if isinstance(obj, dict):
        for wk in ("data", "documents", "examples", "cases", "items", "train", "dev", "test", "validation"):
            if wk in obj and isinstance(obj[wk], (list, dict)) and len(obj) <= 4: return normalise(obj[wk])
        out = []
        for k, v in obj.items():
            if isinstance(v, dict): r = dict(v); r.setdefault("id", k)
            elif isinstance(v, str): r = {"id": k, "text": v}
            else: continue
            out.append(r)
        return out
    return [r for r in obj if isinstance(r, dict)]
def detect(recs):
    keys = list(recs[0].keys()); strlen = {k: np.mean([len(str(r.get(k, ""))) for r in recs[:200]]) for k in keys if isinstance(recs[0].get(k), str)}
    tk = next((k for k in TEXT_KEYS if k in keys), None) or (max(strlen, key=strlen.get) if strlen else None); lk = next((k for k in LABEL_KEYS if k in keys), None)
    if lk is None:
        for k in keys:
            if k == tk: continue
            vals = {str(r.get(k)) for r in recs[:2000] if r.get(k) is not None}
            if 1 < len(vals) <= 5 and k not in ID_KEYS: lk = k; break
    return tk, lk, next((k for k in ID_KEYS if k in keys), None)
def binary_map(vals):
    s = {str(v).strip().lower() for v in vals if v is not None}
    if s <= {"0", "1", "0.0", "1.0", "true", "false"}: return lambda v: int(str(v).strip().lower() in ("1", "1.0", "true"))
    raise ValueError(f"label values {sorted(s)[:10]} are not 0/1: tell me what they mean (for example accepted/rejected) before training")

from tokenizers import Tokenizer
tpaths = sorted(glob.glob("/kaggle/input/**/legal_bpe_32k_mask.json", recursive=True)) or sorted(glob.glob("/kaggle/input/**/legal_bpe_32k*.json", recursive=True)) or sorted(glob.glob("/kaggle/working/**/legal_bpe_32k*.json", recursive=True))
assert tpaths, "attach the tokenizer file legal_bpe_32k_mask.json (it is inside your Phase 1 dataset)"
tok = Tokenizer.from_file(tpaths[0]); EOS, BOS, PAD = tok.token_to_id("[EOS]"), tok.token_to_id("[BOS]"), tok.token_to_id("[PAD]"); print("tokenizer:", tpaths[0], "| vocabulary", tok.get_vocab_size())

def prepare(split, path, bs=256):
    cache = WORK / "cache" / f"{split}.pkl"
    if cache.exists():
        d = pickle.load(open(cache, "rb")); print(f"[{split}] loaded cache: {len(d['tails'])} cases"); return d
    print(f"[{split}] reading {path} ..."); obj = json.load(open(path, encoding="utf-8")); recs = normalise(obj); del obj; gc.collect()
    tk, lk, ik = detect(recs); lmap = binary_map([r.get(lk) for r in recs[:5000]]) if lk else None
    print(f"[{split}] {len(recs)} cases | text field '{tk}' | label field '{lk}' | id field '{ik}'")
    tails, lens, labels, rids = [], [], [], []
    for i in range(0, len(recs), bs):
        chunk = recs[i:i + bs]
        for e in tok.encode_batch([str(r.get(tk, "")) for r in chunk], add_special_tokens=False):
            lens.append(len(e.ids)); tails.append(np.asarray((e.ids + [EOS])[-TAIL_MAX:], dtype=np.int32))
        labels += [lmap(r.get(lk)) if lmap else -1 for r in chunk]; rids += [r.get(ik, i + j) if ik else i + j for j, r in enumerate(chunk)]
        for r in chunk: r.clear()
    d = dict(tails=tails, lens=np.asarray(lens), labels=np.asarray(labels), rids=rids, keys=dict(text=tk, label=lk, id=ik)); pickle.dump(d, open(cache, "wb"), protocol=4); del recs; gc.collect(); return d
DATA = {sp: prepare(sp, p) for sp, p in paths.items()}
for sp, d in DATA.items():
    y = d["labels"]; print(f"[{sp}] cases {len(d['tails']):6d} | tokens p50/p90/p99: {int(np.percentile(d['lens'],50))}/{int(np.percentile(d['lens'],90))}/{int(np.percentile(d['lens'],99))} | label counts: {dict(Counter(y.tolist())) if (y>=0).any() else 'none (test)'}")
assert len(set(map(str, DATA["test"]["rids"]))) == len(DATA["test"]["rids"]), "duplicate test ids"
ytr = DATA["train"]["labels"]; print(f"majority-class baseline macro-F1 on dev would be about {0.5 * (2*max(np.mean(DATA['dev']['labels']==0), np.mean(DATA['dev']['labels']==1)) / (1+max(np.mean(DATA['dev']['labels']==0), np.mean(DATA['dev']['labels']==1)))):.3f}")


In [ ]:
# ================= CELL 3 — load Phase 1 models (all four architectures + the old GPT-2), document classifier, chunked batching =================
al_paths = sorted(glob.glob("/kaggle/input/**/arch_loaders.py", recursive=True)) or sorted(glob.glob("/kaggle/working/**/arch_loaders.py", recursive=True))
assert al_paths, "attach the dataset/notebook output that contains arch_loaders.py (written by cell 8 of the pre-training notebooks)"
sys.path.insert(0, str(Path(al_paths[0]).parent)); import arch_loaders as AL

MODEL_PATHS = {}                                    # filled automatically below; you can also set entries by hand, e.g. MODEL_PATHS["bilstm"] = "/kaggle/input/.../final_model.pt"
def discover_models():
    found = {}
    for p in sorted(glob.glob("/kaggle/input/**/*.pt", recursive=True)):
        if os.path.getsize(p) < 50e6: continue
        base = os.path.basename(p)
        if base != "final_model.pt" and not (base == "checkpoint_latest.pt" and "phase-1" in p): continue
        try: ck = torch.load(p, map_location="cpu", weights_only=False, mmap=True)
        except Exception: continue
        if not isinstance(ck, dict) or "model_config" not in ck or "model_state_dict" not in ck: continue
        mc = ck["model_config"]; fam = mc.get("family", "gpt2"); old = (base == "checkpoint_latest.pt")
        key = "gpt2_old" if old else ("gpt2_new" if fam == "gpt2" else fam)
        if key not in found: found[key] = p
    return found
MODEL_PATHS.update({k: v for k, v in discover_models().items() if k not in MODEL_PATHS})
print("Phase 1 models found:", {k: v for k, v in MODEL_PATHS.items()}, "\n  missing (pre-train them first):", [k for k in ("gpt2_new", "modernbert", "t5", "bilstm", "gpt2_old") if k not in MODEL_PATHS])

def load_backbone_any(path):
    """like AL.load_backbone, but also accepts the OLD Phase 1 checkpoint (its model_config has no 'family' entry)"""
    ck = torch.load(path, map_location="cpu", weights_only=False); mc = dict(ck["model_config"]); mc.setdefault("family", "gpt2")
    full = AL.build_full(mc); full.load_state_dict(ck["model_state_dict"], strict=True); f = mc["family"]
    if f == "gpt2": inner, embed = full.transformer, full.transformer.wte
    elif f == "modernbert": inner, embed = full.model, full.model.embeddings.tok_embeddings
    elif f == "t5": inner, embed = full.encoder, full.shared
    elif f == "bilstm": inner, embed = full, full.emb
    else: raise ValueError(f)
    del ck; return AL.Backbone(f, inner, embed), mc["hidden_size"], AL.CAUSAL[f], mc["context_length"], f

class DocClassifier(nn.Module):
    """Encodes K chunks of a case (the last K*L tokens, right-aligned) with a Phase 1 backbone, pools each chunk, combines the chunks with attention pooling
       + the final chunk, and classifies. K=1 is the plain 'last L tokens' model. Chunks that are empty (short cases) are skipped, never fed to the backbone."""
    def __init__(self, backbone, hidden, causal, K, dropout=0.1):
        super().__init__(); self.backbone, self.hidden, self.causal, self.K = backbone, hidden, causal, K
        self.chunk_pos = nn.Parameter(torch.zeros(K, hidden)); self.proj = nn.Linear(hidden, hidden); self.att = nn.Linear(hidden, 1)
        self.head = nn.Sequential(nn.Dropout(dropout), nn.Linear(2 * hidden, hidden), nn.GELU(), nn.Dropout(dropout), nn.Linear(hidden, 2))
    def encode_chunks(self, ids, mask):
        B, K, L = ids.shape; valid = mask.sum(-1) > 0; fi, fm = ids[valid], mask[valid]; T = int(fm.sum(1).max()); fi, fm = fi[:, :T], fm[:, :T]
        h = self.backbone(input_ids=fi, attention_mask=fm).last_hidden_state
        if self.causal: pooled = h[torch.arange(h.shape[0], device=h.device), fm.sum(1) - 1]
        else: m = fm.unsqueeze(-1).to(h.dtype); pooled = (h * m).sum(1) / m.sum(1).clamp(min=1)
        out = pooled.new_zeros(B, K, self.hidden); out[valid] = pooled; return out, valid
    def forward(self, ids, mask):
        x, valid = self.encode_chunks(ids, mask); x = x + self.chunk_pos.to(x.dtype)
        s = self.att(torch.tanh(self.proj(x))).squeeze(-1).float().masked_fill(~valid, -1e4); w = torch.softmax(s, -1).to(x.dtype)
        return self.head(torch.cat([(w.unsqueeze(-1) * x).sum(1), x[:, -1]], -1))

def make_batch(tails, idx, K, L):
    """last K*L tokens of each case, split RIGHT-aligned into K chunks of <= L tokens (chunk K-1 is the very end); right-padded; returns ids/mask [B,K,L]"""
    B = len(idx); ids = np.full((B, K, L), PAD, dtype=np.int64); mask = np.zeros((B, K, L), dtype=np.int64)
    for b, i in enumerate(idx):
        t = tails[i][-K * L:]; n = len(t)
        for k in range(K - 1, -1, -1):
            end = n - (K - 1 - k) * L
            if end <= 0: break
            seg = t[max(0, end - L):end]; ids[b, k, :len(seg)] = seg; mask[b, k, :len(seg)] = 1
    return torch.from_numpy(ids), torch.from_numpy(mask)


In [ ]:
# ================= CELL 4 — training, evaluation, threshold tuning, submission files =================
import contextlib
USE_BF16 = device.type == "cuda" and torch.cuda.get_device_capability(0)[0] >= 8
def amp_ctx():
    if not AMP: return contextlib.nullcontext()
    return torch.autocast("cuda", dtype=torch.bfloat16 if USE_BF16 else torch.float16)
DEFAULT_LR = {"gpt2": 5e-5, "modernbert": 8e-5, "t5": 1e-4, "bilstm": 1e-3}; HEAD_LR = 1e-3

def prf(y, pred):
    f1s, ps, rs = [], [], []
    for c in (0, 1):
        tp = int(((pred == c) & (y == c)).sum()); fp = int(((pred == c) & (y != c)).sum()); fn = int(((pred != c) & (y == c)).sum())
        p = tp / (tp + fp) if tp + fp else 0.0; r = tp / (tp + fn) if tp + fn else 0.0; f = 2 * p * r / (p + r) if p + r else 0.0; ps.append(p); rs.append(r); f1s.append(f)
    return dict(macro_f1=float(np.mean(f1s)), macro_p=float(np.mean(ps)), macro_r=float(np.mean(rs)), f1_0=f1s[0], f1_1=f1s[1])
def best_threshold(y, prob):
    best_f, best_t = -1.0, 0.5
    for t in sorted(np.arange(0.02, 0.981, 0.01), key=lambda t: abs(t - 0.5)):          # ties -> the threshold closest to 0.5
        f = prf(y, (prob >= t).astype(int))["macro_f1"]
        if f > best_f + 1e-12: best_f, best_t = f, float(t)
    return best_t, best_f

@torch.no_grad()
def predict_probs(model, tails, K, L, tok_budget=16384):
    model.eval(); n = len(tails); order = np.argsort([-len(t) for t in tails]); mb = max(1, tok_budget // (K * L)); probs = np.zeros(n, dtype=np.float32)
    for i in range(0, n, mb):
        idx = order[i:i + mb]; ids, mask = make_batch(tails, idx, K, L)
        with amp_ctx(): logits = model(ids.to(device), mask.to(device)).float()
        probs[idx] = torch.softmax(logits, -1)[:, 1].cpu().numpy()
    return probs

def to_py(x): return x.item() if hasattr(x, "item") else x
def write_submission(path, rids, preds):
    json.dump([{"id": to_py(r), "pred": int(p)} for r, p in zip(rids, preds)], open(path, "w"))
def validate_submission(path, rids):
    sub = json.load(open(path)); assert isinstance(sub, list) and len(sub) == len(rids), f"{path}: expected {len(rids)} predictions, found {len(sub)}"
    for s, r in zip(sub, rids):
        assert set(s) == {"id", "pred"}, f"bad keys {set(s)}"; assert type(s["pred"]) is int and s["pred"] in (0, 1), f"pred must be the integer 0 or 1, got {s['pred']!r}"; assert s["id"] == to_py(r), f"id mismatch {s['id']!r} vs {to_py(r)!r}"
    assert len({str(s['id']) for s in sub}) == len(sub), "duplicate ids"; return True

def run_experiment(name, model_key, K=1, L=512, minutes=30.0, lr=None, bs=16, wd=0.01, warmup=0.06, max_train_docs=None, dev_cap=None, test_cap=None, evals=3, write=True, seed=SEED, dropout=0.1):
    t_start = time.time(); seed_all(seed); rng = np.random.default_rng(seed)
    assert model_key in MODEL_PATHS, f"'{model_key}' is not available: pre-train it first (found: {list(MODEL_PATHS)})"
    bb, hidden, causal, ctx, fam = load_backbone_any(MODEL_PATHS[model_key]); L = min(L, ctx); lr = lr or DEFAULT_LR[fam]
    model = DocClassifier(bb, hidden, causal, K, dropout).to(device); n_params = sum(p.numel() for p in model.parameters())
    tr, dv, te = DATA["train"], DATA["dev"], DATA["test"]
    tr_idx = np.arange(len(tr["tails"])) if not max_train_docs else rng.choice(len(tr["tails"]), min(max_train_docs, len(tr["tails"])), replace=False)
    dv_idx = np.arange(len(dv["tails"])) if not dev_cap else rng.choice(len(dv["tails"]), min(dev_cap, len(dv["tails"])), replace=False); te_idx = np.arange(len(te["tails"])) if not test_cap else np.arange(min(test_cap, len(te["tails"])))
    dv_tails = [dv["tails"][i] for i in dv_idx]; y_dev = dv["labels"][dv_idx]; te_tails = [te["tails"][i] for i in te_idx]
    sub_n = min(1500, len(dv_idx)); sub = rng.choice(len(dv_idx), sub_n, replace=False); sub_tails = [dv_tails[i] for i in sub]; y_sub = y_dev[sub]
    cnt = np.bincount(tr["labels"][tr_idx], minlength=2).astype(float); cw = torch.tensor(cnt.sum() / (2 * np.maximum(cnt, 1)), dtype=torch.float32, device=device)
    mb = max(1, 8192 // (K * L)); accum = max(1, math.ceil(bs / mb)); docs_per_step = mb * accum
    groups = [{"params": list(model.backbone.parameters()), "lr": lr}, {"params": [p for n, p in model.named_parameters() if not n.startswith("backbone.")], "lr": HEAD_LR}]
    opt = torch.optim.AdamW(groups, weight_decay=wd); base_lrs = [g["lr"] for g in groups]; scaler = torch.amp.GradScaler("cuda", enabled=AMP and not USE_BF16)
    def micro(ids_idx):
        ids, mask = make_batch(tr["tails"], ids_idx, K, L); y = torch.from_numpy(tr["labels"][ids_idx]).to(device)
        with amp_ctx(): logits = model(ids.to(device), mask.to(device))
        return F.cross_entropy(logits.float(), y, weight=cw)
    model.train(); times = []
    for j in range(4):                                                            # throughput probe (forward+backward only, no optimizer step); the first is warm-up
        t0 = time.time(); loss = micro(rng.choice(tr_idx, mb)); scaler.scale(loss).backward(); opt.zero_grad(set_to_none=True)
        if device.type == "cuda": torch.cuda.synchronize()
        times.append(time.time() - t0)
    micro_s = float(np.mean(times[1:])); sec_step = micro_s * accum; eval_docs = evals * sub_n + len(dv_tails) + len(te_tails); eval_s = eval_docs * micro_s / mb / 3.0
    steps = max(10, int((minutes * 60 * 0.92 - eval_s) / sec_step)); print(f"[{name}] {fam} | K={K} L={L} | {n_params/1e6:.0f}M params | micro-batch {mb} x accum {accum} = {docs_per_step} cases/step | {sec_step:.2f}s/step -> {steps} steps "
          f"({steps*docs_per_step} cases = {steps*docs_per_step/len(tr_idx):.2f} epochs of {len(tr_idx)}) + ~{eval_s/60:.1f} min of evaluation")
    perm, ptr, best, snap, evals_at, hist = rng.permutation(tr_idx), 0, (-1.0, None), None, {int(round(steps * (i + 1) / evals)) for i in range(evals)}, []
    run_loss = 0.0
    for step in range(1, steps + 1):
        w = min(step / max(1, int(warmup * steps)), (steps - step + 1) / max(1, steps - int(warmup * steps)), 1.0)
        for g, b in zip(opt.param_groups, base_lrs): g["lr"] = b * w
        for _ in range(accum):
            if ptr + mb > len(perm): perm, ptr = rng.permutation(tr_idx), 0
            loss = micro(perm[ptr:ptr + mb]); ptr += mb; scaler.scale(loss / accum).backward(); run_loss += loss.item() / accum
        scaler.unscale_(opt); torch.nn.utils.clip_grad_norm_(model.parameters(), 1.0); scaler.step(opt); scaler.update(); opt.zero_grad(set_to_none=True)
        if step % max(1, steps // 10) == 0: print(f"   step {step}/{steps} | train loss {run_loss/max(1, steps//10):.4f} | {(time.time()-t_start)/60:.1f} min", flush=True); run_loss = 0.0
        if step in evals_at:
            p = predict_probs(model, sub_tails, K, L); t, f = best_threshold(y_sub, p); hist.append((step, round(f, 4))); print(f"   [dev subset of {sub_n}] step {step}: macro-F1 {f:.4f} at threshold {t:.2f}", flush=True); model.train()
            if f > best[0]: best = (f, step); snap = {k: v.detach().to("cpu", copy=True) for k, v in model.state_dict().items()}
    model.load_state_dict(snap); p_dev = predict_probs(model, dv_tails, K, L); p_te = predict_probs(model, te_tails, K, L); thr, f_t = best_threshold(y_dev, p_dev)
    at05 = prf(y_dev, (p_dev >= 0.5).astype(int)); at_t = prf(y_dev, (p_dev >= thr).astype(int))
    res = dict(name=name, model=model_key, family=fam, K=K, L=L, params_M=round(n_params / 1e6, 1), steps=steps, cases_seen=steps * docs_per_step, best_eval_step=best[1], threshold=thr, dev_macro_f1=at_t["macro_f1"], dev_macro_f1_at_0p5=at05["macro_f1"],
               dev_macro_p=at_t["macro_p"], dev_macro_r=at_t["macro_r"], dev_cases=len(y_dev), minutes=round((time.time() - t_start) / 60, 1), history=str(hist))
    print(f"[{name}] DEV macro-F1 {at_t['macro_f1']:.4f} (threshold {thr:.2f}; at 0.5: {at05['macro_f1']:.4f}) | macro-P {at_t['macro_p']:.4f} macro-R {at_t['macro_r']:.4f} | {res['minutes']} min")
    if write:
        rd = WORK / "runs" / name; rd.mkdir(exist_ok=True); np.save(rd / "dev_probs.npy", p_dev); np.save(rd / "test_probs.npy", p_te); json.dump(res, open(rd / "result.json", "w"), indent=1)
        rids = [te["rids"][i] for i in te_idx]; write_submission(SUBDIR / f"{name}.json", rids, (p_te >= thr).astype(int)); validate_submission(SUBDIR / f"{name}.json", rids)
        row = pd.DataFrame([res]); led = pd.concat([pd.read_csv(LEDGER).query('name != @name'), row]) if LEDGER.exists() else row; led.to_csv(LEDGER, index=False)
        print(f"   saved {SUBDIR / (name + '.json')} ({len(rids)} predictions, format validated) and the ledger row")
    del model, opt, bb, snap; gc.collect()
    if device.type == "cuda": torch.cuda.empty_cache()
    return res


In [ ]:
# ================= CELL 5 — SMOKE TEST (~3 min): small data, one architecture, writes nothing. Must pass before the real experiments. =================
res = run_experiment("smoke", "gpt2_new", K=1, L=512, minutes=1.5, max_train_docs=400, dev_cap=300, test_cap=40, evals=1, write=False)
assert np.isfinite(res["dev_macro_f1"]); print("\nsmoke test OK. Throughput above tells you how many minutes each experiment below needs; expect a LOW F1 here (only ~400 training cases).")


In [ ]:
# ================= CELL 6 — THE SIX EXPERIMENTS (run each in its own cell; each writes a validated submission file and a ledger row) =================
# Each experiment trains for a fixed number of MINUTES (the schedule is sized to your real GPU speed). Raise `minutes` for the experiments that matter most.
# An experiment whose model is not available yet (not pre-trained) stops with a clear message: run the others first.
EXPERIMENTS = [
    # name,                 model key,    K (chunks), L (tokens per chunk), minutes
    ("E1_encoder_hier4",    "modernbert", 4,          512,                  45),   # encoder-only, last 4 chunks combined (hierarchical)
    ("E2_encoder_tail512",  "modernbert", 1,          512,                  30),   # encoder-only, last 512 tokens
    ("E3_gpt2_tail512",     "gpt2_new",   1,          512,                  30),   # decoder-only (new Phase 1 model), last 512 tokens
    ("E4_t5enc_tail512",    "t5",         1,          512,                  30),   # encoder of the encoder-decoder model, last 512 tokens
    ("E5_bilstm_tail2048",  "bilstm",     1,          2048,                 30),   # recurrent model reads the last 2,048 tokens natively
    ("E6_gpt2old_tail2048", "gpt2_old",   1,          2048,                 40),   # old 3,000-step GPT-2 (trained on 4,096-token blocks): longer context
]
RESULTS = {}
def run(i): n, m, K, L, mins = EXPERIMENTS[i]; RESULTS[n] = run_experiment(n, m, K=K, L=L, minutes=mins); return RESULTS[n]
for k, e in enumerate(EXPERIMENTS): print(k, e)


In [ ]:
run(2)      # E3: decoder-only (new GPT-2). Start here: this model exists already.


In [ ]:
run(5)      # E6: old 3,000-step GPT-2 with a 2,048-token window


In [ ]:
run(1)      # E2: encoder, last 512 tokens          (needs the pre-trained encoder)
run(0)      # E1: encoder, 4 chunks (hierarchical)


In [ ]:
run(3)      # E4: T5 encoder                         (needs the pre-trained encoder-decoder)


In [ ]:
run(4)      # E5: BiLSTM                             (needs the pre-trained recurrent model)


In [ ]:
# ================= CELL 7 — choose the six submissions, validate them, and package =================
led = pd.read_csv(LEDGER).sort_values("dev_macro_f1", ascending=False); cols = ["name", "model", "K", "L", "dev_macro_f1", "dev_macro_f1_at_0p5", "dev_macro_p", "dev_macro_r", "threshold", "steps", "minutes"]
print(led[cols].round(4).to_string(index=False))
FINAL = None                                   # e.g. FINAL = ["E1_encoder_hier4", "E3_gpt2_tail512", ...] to choose by hand; None = best six, forcing every available model family to appear
if FINAL is None:
    best_per_model = led.sort_values("dev_macro_f1", ascending=False).drop_duplicates("model"); FINAL = list(best_per_model.head(6).name)       # best experiment of each distinct Phase 1 model first
    for n in led.name:
        if len(FINAL) >= 6: break
        if n not in FINAL: FINAL.append(n)
assert len(FINAL) == 6, f"need six experiments, have {len(FINAL)} in the ledger: run more experiments"
fams = led.set_index("name").loc[FINAL, "model"].tolist(); print("\nFINAL SIX:", FINAL, "\nPhase 1 models covered:", sorted(set(fams)))
if not {"gpt2_new", "modernbert", "t5", "bilstm"} <= set(fams): print("WARNING: the rules require experiments with ALL FOUR Phase 1 architectures; missing:", sorted({"gpt2_new", "modernbert", "t5", "bilstm"} - set(fams)))
FINAL_DIR = WORK / "final_six"; FINAL_DIR.mkdir(exist_ok=True); rids = DATA["test"]["rids"]
for k, n in enumerate(FINAL, 1):
    src = SUBDIR / f"{n}.json"; validate_submission(src, rids); (FINAL_DIR / f"submission_{k}_{n}.json").write_text(src.read_text())
# optional extra: probability-average ensemble of the six (tuned on dev); it is NOT one of the six unless you decide to swap it in
P_dev = np.mean([np.load(WORK / "runs" / n / "dev_probs.npy") for n in FINAL], 0); P_te = np.mean([np.load(WORK / "runs" / n / "test_probs.npy") for n in FINAL], 0); y_dev = DATA["dev"]["labels"]
t_ens, f_ens = best_threshold(y_dev, P_dev); write_submission(FINAL_DIR / "extra_ensemble_of_six.json", rids, (P_te >= t_ens).astype(int)); validate_submission(FINAL_DIR / "extra_ensemble_of_six.json", rids)
print(f"ensemble of the six: dev macro-F1 {f_ens:.4f} (best single: {led.dev_macro_f1.max():.4f})")
with zipfile.ZipFile(WORK / "cjpe_submissions.zip", "w", zipfile.ZIP_DEFLATED) as z:
    for p in sorted(FINAL_DIR.glob("*.json")): z.write(p, p.name)
print("\nvalidated:", sorted(p.name for p in FINAL_DIR.glob("*.json")), f"\n{len(rids)} test cases each; format [{{'id': ..., 'pred': 0|1}}]\npackage:", WORK / "cjpe_submissions.zip")
print("IMPORTANT: dev macro-F1 above is tuned on the dev set (threshold, snapshot); the private test score will typically be a little lower.")
